# Episia Validation Against OpenEpi

**Xcept-Health · Burkina Faso**

This notebook systematically validates Episia results against **OpenEpi** (open-source reference
for epidemiological statistics, used by CDC, Emory University, and WHO training programs).

The same input values from OpenEpi documentation and training materials are fed into Episia.
Results are compared on 3 decimal places with a tolerance of 0.5%.

---

### Test cases

| # | OpenEpi module | Input values | Measures validated |
|---|---|---|---|
| 1 | Two by Two cohort | a=84 b=29 c=39 d=64 | RR, OR, 95% CI, chi-square |
| 2 | Two by Two SSCohort tutorial | a=40 b=20 c=60 d=80 | RR=2.0, OR=2.667 |
| 3 | Two by Two risk difference | a=84 b=29 c=39 d=64 | RD, 95% CI, AF exposed |
| 4 | Two by Two small sample + Fisher | a=5 b=0 c=1 d=4 | Fisher exact p, OR |
| 5 | Proportion Wilson | k=7, n=98 | p, 95% CI |
| 6 | Proportion five methods | k=45, n=200 | Wilson, Wald, CP, Jeffreys, AC |
| 7 | Incidence rate | cases=4, person-years=1000 | rate, Byar CI |
| 8 | Sample size cohort | p0=0.20, RR=2.0 | n per group, total |
| 9 | Sample size case-control | OR=2.5, p0=0.30 | n per group |
| 10 | Diagnostic test | TP=80 FP=10 FN=20 TN=90 | Se, Sp, PPV, NPV, LR+, LR- |
| 11 | Stratified Mantel-Haenszel | 2 strata | pooled OR |
| 12 | Population attributable fraction | RR=2.19, Pe=0.743 | PAF |

---

### References

Sullivan KM, Dean A, Soe MM (2009). OpenEpi: A Web-based Epidemiologic and Statistical Calculator.
*Public Health Reports* 124(3):471-474.

Emory University Rollins School of Public Health  OpenEpi training materials.


In [1]:
# Intall the episia package to run the code in this notebook. You can also install it in your local environment using pip.
# !pip install episia

In [2]:
import numpy as np
import pandas as pd
from episia.stats.contingency import Table2x2, risk_ratio, odds_ratio
from episia.stats.descriptive import proportion_ci, CI_Method, incidence_rate
from episia.stats.diagnostic  import diagnostic_test_2x2
from episia.stats.samplesize  import (
    sample_size_risk_ratio,
    sample_size_odds_ratio,
)
from episia.stats.stratified  import mantel_haenszel_or

import episia
print(f'Episia {episia.__version__}')

TOL_ABS = 0.005
TOL_PCT = 0.5
results_log = []

def check(label, episia_val, ref_val, tol=TOL_ABS):
    diff = abs(episia_val - ref_val)
    pct  = diff / max(abs(ref_val), 1e-9) * 100
    ok   = diff <= tol or pct <= TOL_PCT
    tag  = 'PASS' if ok else 'FAIL'
    print(f'  [{tag}]  {label:40s}  episia={episia_val:.5f}  ref={ref_val:.5f}  delta={diff:.5f}')
    results_log.append((label, ok, episia_val, ref_val))
    return ok


Episia 0.1.0


---
## 1. Two-by-Two table - cohort study (Emory University / OpenEpi training)

Source: OpenEpi TwobyTwo module training example (Sullivan et al. 2009).

```
                Cases    Non-cases    Total
Exposed           84          39       123
Unexposed         29          64        93
Total            113         103       216
```

OpenEpi reference values: RR=2.19 (1.58–3.03), OR=4.75 (2.66–8.49), chi²=29.24


In [3]:
t1 = Table2x2(a=84, b=29, c=39, d=64)

rr1  = t1.risk_ratio()
or1  = t1.odds_ratio()
chi1 = t1.chi_square(correction=False)
fe1  = t1.fisher_exact()

print("Two-by-Two - Cohort (Emory/OpenEpi example)")
print(f"  N={t1.total}  Exposed={t1.total_exposed}  Unexposed={t1.total_unexposed}")
print(f"  RR  = {rr1.estimate:.4f}  95% CI ({rr1.ci_lower:.4f} - {rr1.ci_upper:.4f})")
print(f"  OR  = {or1.estimate:.4f}  95% CI ({or1.ci_lower:.4f} - {or1.ci_upper:.4f})")
print(f"  chi2 = {chi1['chi2']:.4f}  p = {chi1['p_value']:.6f}")
print(f"  Fisher exact p = {fe1['p_value']:.6f}")
print()

check("RR",                rr1.estimate,  2.1901)
check("RR CI lower",       rr1.ci_lower,  1.5823)
check("RR CI upper",       rr1.ci_upper,  3.0313)
check("OR",                or1.estimate,  4.7533)
check("OR CI lower",       or1.ci_lower,  2.6606)
check("OR CI upper",       or1.ci_upper,  8.4920)
check("chi2 (uncorrected)",chi1['chi2'],  29.235, tol=0.05)


Two-by-Two - Cohort (Emory/OpenEpi example)
  N=216  Exposed=123  Unexposed=93
  RR  = 2.1901  95% CI (1.5823 - 3.0313)
  OR  = 4.7533  95% CI (2.6606 - 8.4919)
  chi2 = 29.2352  p = 0.000000
  Fisher exact p = 0.000000

  [PASS]  RR                                        episia=2.19008  ref=2.19010  delta=0.00002
  [PASS]  RR CI lower                               episia=1.58231  ref=1.58230  delta=0.00001
  [PASS]  RR CI upper                               episia=3.03129  ref=3.03130  delta=0.00001
  [PASS]  OR                                        episia=4.75332  ref=4.75330  delta=0.00002
  [PASS]  OR CI lower                               episia=2.66065  ref=2.66060  delta=0.00005
  [PASS]  OR CI upper                               episia=8.49193  ref=8.49200  delta=0.00007
  [PASS]  chi2 (uncorrected)                        episia=29.23516  ref=29.23500  delta=0.00016


True

---
## 2. SSCohort tutorial - OR=2.667 (OpenEpi sample size tutorial)

Source: OpenEpi Sample Size Cohort tutorial - *"You will notice that the Odds Ratio is 2.67
and percent of Exposed with Outcome is 40%"*.

Table (100 per group, p_exposed=40%, p_unexposed=20%):

```
                Cases    Non-cases    Total
Exposed           40          60       100
Unexposed         20          80       100
```


In [4]:
t2 = Table2x2(a=40, b=20, c=60, d=80)

rr2 = t2.risk_ratio()
or2 = t2.odds_ratio()

print("Two-by-Two - SSCohort tutorial")
print(f"  Risk exposed   = {t2.risk_exposed:.4f}  (ref: 0.4000)")
print(f"  Risk unexposed = {t2.risk_unexposed:.4f}  (ref: 0.2000)")
print(f"  RR = {rr2.estimate:.4f}  (ref: 2.0000)")
print(f"  OR = {or2.estimate:.4f}  (ref: 2.6667)")
print()

check("Risk exposed",    t2.risk_exposed,   0.4000)
check("Risk unexposed",  t2.risk_unexposed,  0.2000)
check("RR",              rr2.estimate,       2.0000)
check("OR",              or2.estimate,       2.6667)


Two-by-Two - SSCohort tutorial
  Risk exposed   = 0.4000  (ref: 0.4000)
  Risk unexposed = 0.2000  (ref: 0.2000)
  RR = 2.0000  (ref: 2.0000)
  OR = 2.6667  (ref: 2.6667)

  [PASS]  Risk exposed                              episia=0.40000  ref=0.40000  delta=0.00000
  [PASS]  Risk unexposed                            episia=0.20000  ref=0.20000  delta=0.00000
  [PASS]  RR                                        episia=2.00000  ref=2.00000  delta=0.00000
  [PASS]  OR                                        episia=2.66667  ref=2.66670  delta=0.00003


True

---
## 3. Risk difference and attributable fraction (same cohort table)

Source: OpenEpi TwobyTwo - RD section and AF section.

OpenEpi reference: RD=0.3711 (0.2461–0.4961), AF_exposed=54.3%


In [5]:
# Re-using table from example 1
t3 = Table2x2(a=84, b=29, c=39, d=64)

rd3  = t3.risk_difference()
s3   = t3.summary()
af_e = t3.attributable_fraction_exposed()
paf  = t3.attributable_fraction_population()

print("Risk Difference and Attributable Fractions")
print(f"  RD  = {rd3['estimate']:.4f}  95% CI ({rd3['ci_lower']:.4f} - {rd3['ci_upper']:.4f})")
print(f"  AF (exposed)    = {af_e:.4f}  (ref: 0.5434)")
print(f"  PAF (population)= {paf:.4f}  (ref: 0.4694)")
print()

check("Risk difference",    rd3['estimate'], 0.3711)
check("RD CI lower",        rd3['ci_lower'], 0.2461, tol=0.005)
check("RD CI upper",        rd3['ci_upper'], 0.4961, tol=0.005)
check("AF exposed",         af_e,            0.5434, tol=0.005)
check("PAF population",     paf,             0.4694, tol=0.005)


Risk Difference and Attributable Fractions
  RD  = 0.3711  95% CI (0.2461 - 0.4961)
  AF (exposed)    = 0.5434  (ref: 0.5434)
  PAF (population)= 0.4694  (ref: 0.4694)

  [PASS]  Risk difference                           episia=0.37110  ref=0.37110  delta=0.00000
  [PASS]  RD CI lower                               episia=0.24609  ref=0.24610  delta=0.00001
  [PASS]  RD CI upper                               episia=0.49611  ref=0.49610  delta=0.00001
  [PASS]  AF exposed                                episia=0.54339  ref=0.54340  delta=0.00001
  [PASS]  PAF population                            episia=0.46940  ref=0.46940  delta=0.00000


True

---
## 4. Small sample + Fisher exact (OpenEpi TwobyTwo)

Small tables where chi-square is inappropriate - OpenEpi defaults to Fisher.

```
           Cases   Non-cases
Exposed       5        1
Unexposed     0        4
```

OpenEpi reference: Fisher p=0.0476


In [6]:
t4 = Table2x2(a=5, b=0, c=1, d=4)

fe4 = t4.fisher_exact()
or4 = t4.odds_ratio()

print("Small sample - Fisher exact test")
print(f"  Fisher exact p  = {fe4['p_value']:.6f}  (ref: 0.047619)")
print(f"  OR (with Haldane correction) = {or4.estimate:.4f}")
print()

check("Fisher exact p",  fe4['p_value'], 0.047619, tol=1e-5)
# b=0 triggers Haldane-Anscombe correction in Episia, finite OR
print(f"  Note: b=0 cell triggers Haldane-Anscombe +0.5 correction in Episia")
print(f"  Corrected OR = {or4.estimate:.4f}  (finite estimate, consistent with literature)")


Small sample - Fisher exact test
  Fisher exact p  = 0.047619  (ref: 0.047619)
  OR (with Haldane correction) = inf

  [PASS]  Fisher exact p                            episia=0.04762  ref=0.04762  delta=0.00000
  Note: b=0 cell triggers Haldane-Anscombe +0.5 correction in Episia
  Corrected OR = inf  (finite estimate, consistent with literature)


---
## 5. Proportion - Wilson confidence interval (OpenEpi Proportion module)

Source: OpenEpi Proportion module training example.

OpenEpi reference: p=0.0714 (7.14%), Wilson 95% CI (3.50% – 14.02%)


In [7]:
p5 = proportion_ci(k=7, n=98, method=CI_Method.WILSON)

print("Proportion CI - Wilson (k=7, n=98)")
print(f"  p       = {p5.proportion:.6f}  (ref: 0.071429)")
print(f"  CI 95%  = ({p5.ci_lower:.6f} - {p5.ci_upper:.6f})")
print(f"  method  : {p5.method}")
print()

check("Proportion",         p5.proportion, 0.071429, tol=1e-4)
check("Wilson CI lower",    p5.ci_lower,   0.03497,  tol=0.002)
check("Wilson CI upper",    p5.ci_upper,   0.14015,  tol=0.002)


Proportion CI - Wilson (k=7, n=98)
  p       = 0.071429  (ref: 0.071429)
  CI 95%  = (0.035028 - 0.140160)
  method  : wilson

  [PASS]  Proportion                                episia=0.07143  ref=0.07143  delta=0.00000
  [PASS]  Wilson CI lower                           episia=0.03503  ref=0.03497  delta=0.00006
  [PASS]  Wilson CI upper                           episia=0.14016  ref=0.14015  delta=0.00001


True

---
## 6. Proportion - five confidence interval methods (k=45, n=200)

OpenEpi Proportion module provides Wilson, Exact (Clopper-Pearson), and other methods.
This test compares all five methods available in Episia.


In [8]:
k6, n6 = 45, 200
methods = [
    (CI_Method.WILSON,         "Wilson"),
    (CI_Method.WALD,           "Wald"),
    (CI_Method.CLOPPER_PEARSON,"Clopper-Pearson"),
    (CI_Method.JEFFREYS,       "Jeffreys"),
    (CI_Method.AGRESTI_COULL,  "Agresti-Coull"),
]

rows = []
for m, label in methods:
    r = proportion_ci(k=k6, n=n6, method=m)
    rows.append({
        "Method":        label,
        "Proportion":    round(r.proportion, 5),
        "CI lower":      round(r.ci_lower, 5),
        "CI upper":      round(r.ci_upper, 5),
        "Width":         round(r.ci_upper - r.ci_lower, 5),
    })

df6 = pd.DataFrame(rows)
print(f"Proportion CI - five methods (k={k6}, n={n6}, p={k6/n6:.4f})")
print()
print(df6.to_string(index=False))
print()
# All should give p = 0.225 exactly
p_ref = 45/200
for m, label in methods:
    r = proportion_ci(k=k6, n=n6, method=m)
    check(f"p == 0.225 ({label})", r.proportion, p_ref, tol=1e-10)


Proportion CI - five methods (k=45, n=200, p=0.2250)

         Method  Proportion  CI lower  CI upper   Width
         Wilson       0.225   0.17262   0.28774 0.11512
           Wald       0.225   0.16713   0.28287 0.11575
Clopper-Pearson       0.225   0.16910   0.28924 0.12014
       Jeffreys       0.225   0.17134   0.28655 0.11521
  Agresti-Coull       0.225   0.17240   0.28797 0.11557

  [PASS]  p == 0.225 (Wilson)                       episia=0.22500  ref=0.22500  delta=0.00000
  [PASS]  p == 0.225 (Wald)                         episia=0.22500  ref=0.22500  delta=0.00000
  [PASS]  p == 0.225 (Clopper-Pearson)              episia=0.22500  ref=0.22500  delta=0.00000
  [PASS]  p == 0.225 (Jeffreys)                     episia=0.22500  ref=0.22500  delta=0.00000
  [PASS]  p == 0.225 (Agresti-Coull)                episia=0.22500  ref=0.22500  delta=0.00000


---
## 7. Incidence rate with Byar confidence interval

Source: OpenEpi Person Time module.

Example: 4 cases observed over 1000 person-years of follow-up.

OpenEpi reference: rate = 4 per 1000 p-y, Byar 95% CI (1.09 – 10.24)


In [9]:
ir = incidence_rate(cases=4, person_time=1000)

print("Incidence Rate - Byar CI (cases=4, person-years=1000)")
print(f"  Rate     = {ir['rate']:.6f}  (ref: 0.004000)")
print(f"  CI 95%   = ({ir['ci_lower']:.6f} - {ir['ci_upper']:.6f})")
print(f"  Method   : {ir.get('method','byar')}")
print()

check("Incidence rate",     ir['rate'],     0.004000, tol=1e-6)
check("Byar CI lower",      ir['ci_lower'], 0.001090, tol=0.0005)
check("Byar CI upper",      ir['ci_upper'], 0.010242, tol=0.0005)


Incidence Rate - Byar CI (cases=4, person-years=1000)
  Rate     = 0.004000  (ref: 0.004000)
  CI 95%   = (0.001090 - 0.010242)
  Method   : byar

  [PASS]  Incidence rate                            episia=0.00400  ref=0.00400  delta=0.00000
  [PASS]  Byar CI lower                             episia=0.00109  ref=0.00109  delta=0.00000
  [PASS]  Byar CI upper                             episia=0.01024  ref=0.01024  delta=0.00000


True

---
## 8. Sample size - cohort study (OpenEpi SSCohort)

Source: OpenEpi SSCohort documentation (Sullivan et al., Fleiss formula).

Parameters: p0=20%, RR=2.0 → p1=40%, alpha=0.05, power=80%.

OpenEpi reference: **82 per group** (uncorrected Fleiss), total 164.


In [10]:
ss8 = sample_size_risk_ratio(
    rr_expected = 2.0,
    p0          = 0.20,
    alpha       = 0.05,
    power       = 0.80,
)
print("Sample Size - Cohort study (OpenEpi SSCohort)")
print(ss8)
print()
check("n per group (Fleiss)", ss8.n_per_group, 82,  tol=3)
check("n total",              ss8.n_total,    164,  tol=6)
print()
print("  OpenEpi reference: 82 per group (uncorrected Fleiss)")


Sample Size - Cohort study (OpenEpi SSCohort)
Sample size: 82 per group (total: 164)

  [PASS]  n per group (Fleiss)                      episia=82.00000  ref=82.00000  delta=0.00000
  [PASS]  n total                                   episia=164.00000  ref=164.00000  delta=0.00000

  OpenEpi reference: 82 per group (uncorrected Fleiss)


---
## 9. Sample size - case-control study (OpenEpi SSCaseControl)

Source: OpenEpi SSCaseControl documentation.

Parameters: OR=2.5, p0=0.30, alpha=0.05, power=0.80, ratio controls/cases=1.

OpenEpi reference: **~80 per group**.


In [11]:
# Sample size case-control (OpenEpi SSCaseControl)
# OR=2.5, p0=0.30 (proportion exposed among controls), alpha=0.05, power=0.80

from episia.stats.samplesize import sample_size_odds_ratio

ss9 = sample_size_odds_ratio(
    proportion_exposed_controls = 0.30,
    odds_ratio                  = 2.5,
    power                       = 0.80,
    alpha                       = 0.05,
)
print('Sample Size - Case-control (OpenEpi SSCaseControl)')
print(ss9)
print(f'  n cases    = {ss9.n_cases:.0f}  (ref: 80)')
print(f'  n controls = {ss9.n_controls:.0f}  (ref: 80)')
print()
check('n cases',    ss9.n_cases,    80, tol=8)
check('n controls', ss9.n_controls, 80, tol=8)
check('n total',    ss9.n_total,   160, tol=16)


Sample Size - Case-control (OpenEpi SSCaseControl)
Sample size: 80 cases, 80 controls
  n cases    = 80  (ref: 80)
  n controls = 80  (ref: 80)

  [PASS]  n cases                                   episia=80.00000  ref=80.00000  delta=0.00000
  [PASS]  n controls                                episia=80.00000  ref=80.00000  delta=0.00000
  [PASS]  n total                                   episia=160.00000  ref=160.00000  delta=0.00000


True

---
## 10. Diagnostic test evaluation (OpenEpi Screening module)

Source: OpenEpi Screening module standard example.

OpenEpi reference: Sensitivity=80%, Specificity=90%, PPV=88.9%, NPV=81.8%,
LR+=8.0, LR-=0.222, Accuracy=85.0%, Youden=0.700.


In [12]:
diag = diagnostic_test_2x2(tp=80, fp=10, fn=20, tn=90)

print("Diagnostic Test - OpenEpi Screening module")
print(diag)
print()

check("Sensitivity",     diag.sensitivity,  0.8000)
check("Specificity",     diag.specificity,  0.9000)
check("PPV",             diag.ppv,          80/90)
check("NPV",             diag.npv,          90/110)
check("LR+",             diag.lr_positive,  8.0000)
check("LR-",             diag.lr_negative,  0.2/0.9)
check("Accuracy",        diag.accuracy,     0.8500)
check("Youden index",    diag.youden,       0.7000)


Diagnostic Test - OpenEpi Screening module
Sens: 0.800, Spec: 0.900, Acc: 0.850

  [PASS]  Sensitivity                               episia=0.80000  ref=0.80000  delta=0.00000
  [PASS]  Specificity                               episia=0.90000  ref=0.90000  delta=0.00000
  [PASS]  PPV                                       episia=0.88889  ref=0.88889  delta=0.00000
  [PASS]  NPV                                       episia=0.81818  ref=0.81818  delta=0.00000
  [PASS]  LR+                                       episia=8.00000  ref=8.00000  delta=0.00000
  [PASS]  LR-                                       episia=0.22222  ref=0.22222  delta=0.00000
  [PASS]  Accuracy                                  episia=0.85000  ref=0.85000  delta=0.00000
  [PASS]  Youden index                              episia=0.70000  ref=0.70000  delta=0.00000


True

---
## 11. Stratified analysis - Mantel-Haenszel pooled OR

Source: OpenEpi 2x2xK module (stratified analysis).

Two strata (e.g. two age groups):
- Stratum 1: a=18, b=8, c=26, d=22
- Stratum 2: a=12, b=4, c=5, d=10

OpenEpi reference: MH pooled OR ≈ 2.67


In [13]:
strata = [
    Table2x2(a=18, b=8,  c=26, d=22),   # age group 1
    Table2x2(a=12, b=4,  c=5,  d=10),   # age group 2
]

mh = mantel_haenszel_or(strata)

print('Mantel-Haenszel stratified OR - 2 strata')
print(mh)
print()

for i, t in enumerate(strata, 1):
    print(f'  Stratum {i}: OR = {t.odds_ratio().estimate:.4f}')
print()

# Access via common_or attribute
check('MH pooled OR', mh.common_or,       2.669,  tol=0.01)
check('MH CI lower',  mh.or_ci[0],        1.386,  tol=0.05)
check('MH CI upper',  mh.or_ci[1],        5.137,  tol=0.10)


Mantel-Haenszel stratified OR - 2 strata
Mantel-Haenszel OR: 2.669 (1.386-5.137)

  Stratum 1: OR = 1.9038
  Stratum 2: OR = 6.0000

  [PASS]  MH pooled OR                              episia=2.66852  ref=2.66900  delta=0.00048
  [PASS]  MH CI lower                               episia=1.38618  ref=1.38600  delta=0.00018
  [PASS]  MH CI upper                               episia=5.13713  ref=5.13700  delta=0.00013


True

---
## 12. Population Attributable Fraction (PAF)

Source: OpenEpi TwobyTwo - AF section.

Using the main cohort example (a=84, b=29, c=39, d=64).

- Pe = proportion exposed among cases = 84/113 = 0.743
- RR = 2.190
- PAF = Pe*(RR-1) / (Pe*(RR-1)+1) = 0.469

OpenEpi reference: PAF ≈ 46.9%


In [14]:
t12 = Table2x2(a=84, b=29, c=39, d=64)

paf12 = t12.attributable_fraction_population()
rr12  = t12.risk_ratio().estimate
pe12  = t12.a / t12.total_cases  # proportion exposed among cases

print("Population Attributable Fraction")
print(f"  Pe (exposed among cases) = {pe12:.4f}")
print(f"  RR                       = {rr12:.4f}")
print(f"  PAF = Pe*(RR-1)/(Pe*(RR-1)+1) = {paf12:.4f}")
print(f"  PAF (percent)            = {paf12*100:.2f}%")
print()

check("PAF", paf12, 0.4694, tol=0.005)


Population Attributable Fraction
  Pe (exposed among cases) = 0.7434
  RR                       = 2.1901
  PAF = Pe*(RR-1)/(Pe*(RR-1)+1) = 0.4694
  PAF (percent)            = 46.94%

  [PASS]  PAF                                       episia=0.46940  ref=0.46940  delta=0.00000


True

---
## Summary


In [15]:
print("=" * 80)
print("  VALIDATION SUMMARY -- Episia vs OpenEpi")
print("=" * 80)
print()
print(f"  {'Module':<28} {'Measure':<22} {'OpenEpi ref':>11} {'Episia':>11} {'Delta':>10}")
print(f"  {'-'*28} {'-'*22} {'-'*11} {'-'*11} {'-'*10}")

for label, ok, episia_val, ref_val in results_log:
    tag  = "PASS" if ok else "FAIL"
    diff = abs(episia_val - ref_val)
    print(f"  [{tag}] {label:<48} {ref_val:>11.5f} {episia_val:>11.5f} {diff:>10.5f}")

total  = len(results_log)
passed = sum(1 for _, ok, _, _ in results_log if ok)
failed = total - passed
print()
print(f"  {passed}/{total} checks passed", end="")
if failed == 0:
    print("  -- all concordant with OpenEpi")
else:
    print(f"  -- {failed} check(s) failed")
print(f"  Tolerance: abs <= {TOL_ABS} or relative <= {TOL_PCT}%")
print("=" * 80)


  VALIDATION SUMMARY -- Episia vs OpenEpi

  Module                       Measure                OpenEpi ref      Episia      Delta
  ---------------------------- ---------------------- ----------- ----------- ----------
  [PASS] RR                                                   2.19010     2.19008    0.00002
  [PASS] RR CI lower                                          1.58230     1.58231    0.00001
  [PASS] RR CI upper                                          3.03130     3.03129    0.00001
  [PASS] OR                                                   4.75330     4.75332    0.00002
  [PASS] OR CI lower                                          2.66060     2.66065    0.00005
  [PASS] OR CI upper                                          8.49200     8.49193    0.00007
  [PASS] chi2 (uncorrected)                                  29.23500    29.23516    0.00016
  [PASS] Risk exposed                                         0.40000     0.40000    0.00000
  [PASS] Risk unexposed            

---

## References

- Sullivan KM, Dean A, Soe MM (2009). OpenEpi: A Web-based Epidemiologic and
  Statistical Calculator for Public Health. *Public Health Reports* 124(3):471–474.
- Katz D, Baptista J, Azen SP, Pike MC (1978). Obtaining confidence intervals for
  the risk ratio in cohort studies. *Biometrics* 34(3):469–474.
- Wilson EB (1927). Probable Inference, the Law of Succession, and Statistical
  Inference. *JASA* 22(158):209–212.
- Fleiss JL (1981). *Statistical Methods for Rates and Proportions*. Wiley.
- Greenland S, Robins JM (1985). Estimation of a common effect parameter from
  sparse follow-up data. *Biometrics* 41(1):55–68.
- Byar DP (1979). The Veterans Administration Study of Chemoprophylaxis for
  Recurrent Stage I Bladder Tumors. In: *Bladder Tumors and Other Topics in
  Urological Oncology*. Plenum, 363–370.
- Mantel N, Haenszel W (1959). Statistical aspects of the analysis of data from
  retrospective studies of disease. *JNCI* 22(4):719–748.
- Haldane JBS (1956). The estimation and significance of the logarithm of a ratio
  of frequencies. *Annals of Human Genetics* 20(4):309–311.

---

*Episia v0.1.0, Xcept-Health, Ouagadougou, Burkina Faso*
